In [2]:
import netCDF4 as nc
import json

In [6]:
ds = nc.Dataset('latest_temperature_data.nc')


In [8]:
ds

<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    dimensions(sizes): time(6), yy(150), xx(300)
    variables(dimensions): float64 T2M(time, yy, xx), float64 XX(yy, xx), float64 YY(yy, xx), float64 DX(yy, xx), float64 DY(yy, xx), float64 LATORI(), float64 LONORI(), float64 LAT0(), float64 LON0()
    groups: 

In [17]:
ds['LON0']

<class 'netCDF4.Variable'>
float64 LON0()
    _FillValue: nan
    long_name: LON0
    comment: 
unlimited dimensions: 
current shape = ()
filling on

In [9]:
ds.variables

{'T2M': <class 'netCDF4.Variable'>
 float64 T2M(time, yy, xx)
     _FillValue: 1e+20
     long_name: T2M
     comment: X_Y_T2M          (K)
 unlimited dimensions: 
 current shape = (6, 150, 300)
 filling on,
 'XX': <class 'netCDF4.Variable'>
 float64 XX(yy, xx)
     _FillValue: 1e+20
     long_name: XX
     comment: 
 unlimited dimensions: 
 current shape = (150, 300)
 filling on,
 'YY': <class 'netCDF4.Variable'>
 float64 YY(yy, xx)
     _FillValue: 1e+20
     long_name: YY
     comment: 
 unlimited dimensions: 
 current shape = (150, 300)
 filling on,
 'DX': <class 'netCDF4.Variable'>
 float64 DX(yy, xx)
     _FillValue: 1e+20
     long_name: DX
     comment: 
 unlimited dimensions: 
 current shape = (150, 300)
 filling on,
 'DY': <class 'netCDF4.Variable'>
 float64 DY(yy, xx)
     _FillValue: 1e+20
     long_name: DY
     comment: 
 unlimited dimensions: 
 current shape = (150, 300)
 filling on,
 'LATORI': <class 'netCDF4.Variable'>
 float64 LATORI()
     _FillValue: nan
     long_n

In [18]:
ds['T2M']

<class 'netCDF4.Variable'>
float64 T2M(time, yy, xx)
    _FillValue: 1e+20
    long_name: T2M
    comment: X_Y_T2M          (K)
unlimited dimensions: 
current shape = (6, 150, 300)
filling on

In [19]:
import json
import numpy as np

# 1. Extract data from the netCDF4 variable
# This loads the (6, 150, 300) array into memory
data = ds['T2M'][:]

# 2. Handle masked values or fill values (e.g., 1e+20)
if np.ma.is_masked(data):
  # Fill masked elements with NaN temporarily
  data = data.filled(np.nan)

# Explicitly replace NetCDF fill values (1e+20 or similar large numbers) with NaN
data = np.where(data >= 1e19, np.nan, data)

# 3. Convert the NumPy array to a nested Python list
data_list = data.tolist()


# 4. Recursively replace NaNs with None (which translates to `null` in JSON)
# Standard JSON doesn't support NaN, so converting to `null` is safest for web/JS use.
def replace_nan_with_none(obj):
  if isinstance(obj, list):
    return [replace_nan_with_none(item) for item in obj]
  elif isinstance(obj, float) and np.isnan(obj):
    return None
  return obj


cleaned_data = replace_nan_with_none(data_list)

# 5. Save to latest_temperature_data.json
with open('latest_temperature_data.json', 'w') as f:
  json.dump(cleaned_data, f)

print('Successfully saved temperature data to latest_temperature_data.json!')

Successfully saved temperature data to latest_temperature_data.json!
